In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))          # lets Python find the src/ folder

from src import scorecard as sc

test = pd.read_parquet(ROOT / "data" / "processed" / "test.parquet")
test_score = sc.score(test)
y = test["TARGET"]
print(len(test), "applicants, mean score", round(test_score.mean()))


(CVXPY) Nov 10 11:19:18 AM: Encountered unexpected exception importing solver HIGHS:
ImportError('/home/fiza/Credit_Risk/.venv/lib/python3.12/site-packages/highspy/_core.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN5Highs13releaseMemoryEv')


92254 applicants, mean score 568


In [2]:
GAIN, LOSS = 1, 5          # business assumption: a bad loan costs 5x what a good loan earns

rows = []
for cut in range(500, 641, 10):
    ok = test_score >= cut                     # approved
    rows.append({
        "cutoff": cut,
        "approval_%":            ok.mean() * 100,
        "bad_rate_approved_%":   y[ok].mean() * 100,
        "defaulters_avoided_%":  (~ok & (y == 1)).sum() / (y == 1).sum() * 100,
        "good_customers_lost_%": (~ok & (y == 0)).sum() / (y == 0).sum() * 100,
        "profit": (ok & (y == 0)).sum() * GAIN - (ok & (y == 1)).sum() * LOSS,
    })

cutoffs = pd.DataFrame(rows).round(1)
approve_all = (y == 0).sum() * GAIN - (y == 1).sum() * LOSS
print("Profit if we approve everyone:", approve_all)
print("Best cutoff:", cutoffs.loc[cutoffs["profit"].idxmax(), "cutoff"])
cutoffs


Profit if we approve everyone: 47566
Best cutoff: 530


,cutoff,approval_%,bad_rate_approved_%,defaulters_avoided_%,good_customers_lost_%,profit
0,500,99.1,7.8,4.9,0.6,48890
1,510,97.7,7.4,10.7,1.6,50179
2,520,95.0,6.8,19.7,3.7,51748
3,530,90.6,6.2,30.9,7.5,52705
4,540,83.5,5.3,44.9,14.0,52392
5,550,74.0,4.5,59.2,23.1,50038
6,560,62.3,3.7,71.7,34.7,44794
7,570,48.9,3.0,82.0,48.4,37070
8,580,35.4,2.4,89.3,62.4,27910
9,590,23.2,1.9,94.5,75.3,18938


In [5]:
import importlib
importlib.reload(sc)          # Python caches imports; reload picks up the file you just edited

CUTOFF = 530
rejected = test[test_score < CUTOFF].head(3)

for (idx, person), reasons in zip(rejected.iterrows(), sc.reason_codes(rejected)):
    s = test_score[idx]
    print(f"Applicant {person['SK_ID_CURR']}: score {s:.0f}, "
          f"default risk {sc.prob_default(s):.1%}, defaulted = {person['TARGET']}")
    for i, r in enumerate(reasons, 1):
        print(f"   {i}. {r}")
    print()


Applicant 455506: score 511, default risk 30.4%, defaulted = 0
   1. Low external credit scores
   2. Late instalment payments in the last 12 months
   3. Previous loan applications were refused

Applicant 353952: score 518, default risk 25.3%, defaulted = 0
   1. Low external credit scores
   2. Number of active loans with other lenders
   3. High outstanding debt relative to credit with other lenders

Applicant 393182: score 521, default risk 23.5%, defaulted = 0
   1. Low external credit scores
   2. High outstanding debt relative to credit with other lenders
   3. Late instalment payments in the last 12 months



In [6]:
all_rejected = test[test_score < CUTOFF]
main_reason = sc.reason_codes(all_rejected, n=1).str[0]
(main_reason.value_counts(normalize=True) * 100).round(1)


Low external credit scores                                     89.6
Late instalment payments in the last 12 months                  5.0
Loan amount high relative to the price of the goods             3.4
High outstanding debt relative to credit with other lenders     1.5
Previous loan applications were refused                         0.4
Previous loans: amount granted vs requested                     0.2
Name: proportion, dtype: float64

In [ ]:
    print()
